# Model v2: Daha Büyük Model + Daha Yüksek Çözünürlük (gece eğitimi)

| | v1 (mevcut) | v2 (bu notebook) |
|---|---|---|
| Model | YOLO11s (9 milyon parametre) | **YOLO11m** (20 milyon parametre) |
| Görüntü boyutu | 640 px | **800 px** |
| Veri | RDD2022, 5 ülke | Aynı RDD verisi + **Türkiye'nin 20 ilinden (25 bölge) görüntüler** (yalnızca eğitim bölümüne) |
| Test | RDD test bölümü | **Aynı** test bölümü (karşılaştırma adil) |

Amaç: Bursa'daki asıl sorun olan **kaçırmayı** azaltmak. Büyük model ve yüksek çözünürlük, uzaktaki küçük çatlak ve çukurları
daha iyi yakalar. Türkiye görüntüleri modele Türk yollarını (şerit çizgileri, parke taşı, asfalt dokusu) tanıtır.
Türkiye verisi yarı otomatik etiketlidir: v1 modelinin çok emin olduğu tespitler ve hiçbir şey görmediği sağlam yollar;
emin olunmayan görüntüler atılmıştır. Bursa bilerek dışarıda bırakıldı: Bursa test seti bağımsız kalsın.

Ek olarak her iki model için **sınıf bazlı güven eşikleri** doğrulama setinde (test setinde değil) hesaplanır.

## Gece çalıştırma talimatı
1. `Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU`
2. `Çalışma zamanı → Tümünü çalıştır`
3. İlk hücrede Google Drive izni istenir: **onaylayın, sonra uyuyabilirsiniz.** Geri kalan her şey sırayla kendiliğinden çalışır.
   Başlamadan önce `turkiye_egitim.zip` dosyasını Drive'daki `yol-hasar-tespit` klasörüne yüklemiş olun.
4. Bilgisayar uyku moduna geçmesin, sekme açık kalsın, şarj kablosu takılı olsun.
5. Oturum koparsa: aynı şekilde `Tümünü çalıştır`. Eğitim kaldığı epoch'tan devam eder.

Sonuçlar Drive'da `yol-hasar-tespit/v2/` klasörüne yazılır.

## 1. Ortam

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU YOK! Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU, sonra baştan çalıştırın.'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/yol-hasar-tespit'
OUT = f'{DRIVE_DIR}/v2'
os.makedirs(OUT, exist_ok=True)

!git clone -q https://github.com/mahmutsamiseyhan/yol-hasar-tespit.git /content/yol-hasar-tespit || (cd /content/yol-hasar-tespit && git pull -q)
%cd /content/yol-hasar-tespit
!pip install -q ultralytics remotezip

## 2. Veri (v1 ile aynı ülkeler ve aynı bölümler, seed=42)

In [ ]:
!python scripts/download_rdd2022.py --out /content/rdd_raw
!python scripts/prepare_rdd2022.py --src /content/rdd_raw --out /content/rdd_yolo
DATA_YAML = '/content/rdd_yolo/data.yaml'

## 2b. Türkiye verisi (yalnızca eğitim bölümüne)

In [ ]:
import zipfile, shutil, json
from pathlib import Path
TR_ZIP = Path(DRIVE_DIR) / 'turkiye_egitim.zip'
if TR_ZIP.exists():
    with zipfile.ZipFile(TR_ZIP) as z:
        z.extractall('/content/turkiye')
    n = 0
    for img in Path('/content/turkiye/images').glob('*.jpg'):
        shutil.copy(img, f'/content/rdd_yolo/images/train/{img.name}')
        shutil.copy(Path('/content/turkiye/labels') / f'{img.stem}.txt', f'/content/rdd_yolo/labels/train/{img.stem}.txt')
        n += 1
    Path('/content/rdd_yolo/labels/train.cache').unlink(missing_ok=True)  # YOLO etiketleri yeniden tarasın
    rapor = json.load(open('/content/turkiye/rapor.json'))
    print(f'Türkiye: {n} görüntü eğitime eklendi.', rapor['kararlar'], rapor['kutular'])
else:
    print("UYARI: turkiye_egitim.zip Drive'da yok; v2 yalnızca RDD verisiyle eğitilecek.")

## 3. Eğitim (v2)
İlk epoch bitince satırdaki süreye bakın: toplam süre ≈ epoch süresi × 60. Model 15 epoch boyunca iyileşmezse erken durur.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

MODEL, IMGSZ, EPOCHS, BATCH = 'yolo11m.pt', 800, 60, 16
RUN_NAME = 'rdd2022_yolo11m_800'
PROJECT = f'{DRIVE_DIR}/runs'

last = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'
if last.exists():
    try:
        print('Önceki eğitim bulundu, kaldığı yerden devam ediliyor:', last)
        model = YOLO(str(last))
        model.train(resume=True)
        RUN_DIR = Path(model.trainer.save_dir)
    except AssertionError as e:  # eğitim zaten bitmişse Ultralytics devam etmeyi reddeder
        print('Eğitim zaten tamamlanmış, sonuçlar kullanılıyor.', e)
        RUN_DIR = last.parent.parent
else:
    model = YOLO(MODEL)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=15,
                project=PROJECT, name=RUN_NAME, exist_ok=True, cos_lr=True, close_mosaic=10,
                seed=42, plots=True, workers=8)
    RUN_DIR = Path(model.trainer.save_dir)
BEST_V2 = RUN_DIR / 'weights' / 'best.pt'
import shutil
shutil.copy(BEST_V2, f'{OUT}/best_v2.pt')
print('Kaydedildi:', f'{OUT}/best_v2.pt')

## 4. Değerlendirme: v1 ile v2, aynı test setinde

- **mAP50 / AP50:** standart nesne tespiti başarı notu (eşikten bağımsız)
- **Sınıf bazlı eşikler:** her sınıf için doğrulama setinde F1'i en yüksek yapan güven eşiği. Uygulama bu eşiklerle karar verir.

In [ ]:
import json, numpy as np, pandas as pd
from IPython.display import display

BEST_V1 = Path(DRIVE_DIR) / 'best.pt'   # ilk notebook'un kaydettiği model
assert BEST_V1.exists(), f'v1 modeli bulunamadı: {BEST_V1}'
MODELS = {'v1 (YOLO11s, 640)': (str(BEST_V1), 640), 'v2 (YOLO11m, 800)': (str(BEST_V2), IMGSZ)}

def f1_thresholds(metrics, names, lo=0.05, hi=0.9, fallback=0.35):
    # Sınıf başına F1'i en yüksek yapan eşik. Güvenlik: [lo, hi] aralığına sıkıştırılır;
    # F1 çok düşükse (sınıf öğrenilememiş) genel eşik kullanılır.
    b = metrics.box
    out = {}
    for i, cls_id in enumerate(b.ap_class_index):
        f1 = np.asarray(b.f1_curve)[i]
        j = int(f1.argmax())
        esik = float(np.asarray(b.px)[j]) if f1[j] >= 0.1 else fallback
        out[names[int(cls_id)]] = {'esik': round(min(hi, max(lo, esik)), 3), 'f1': round(float(f1[j]), 3)}
    return out

results, rows = {}, []
for name, (path, sz) in MODELS.items():
    m = YOLO(path)
    val = m.val(data=DATA_YAML, split='val', imgsz=sz, batch=16, plots=False, verbose=False)
    test = m.val(data=DATA_YAML, split='test', imgsz=sz, batch=16, plots=True, verbose=False,
                 project=OUT, name=name.split()[0] + '_test', exist_ok=True)
    th = f1_thresholds(val, m.names)
    per_class = {}
    for i, cls_id in enumerate(test.box.ap_class_index):
        p, r, ap50, ap = test.box.class_result(i)
        per_class[m.names[int(cls_id)]] = round(float(ap50), 3)
    results[name] = {'mAP50': round(float(test.box.map50), 3), 'mAP50-95': round(float(test.box.map), 3),
                     'precision': round(float(test.box.mp), 3), 'recall': round(float(test.box.mr), 3),
                     'AP50': per_class, 'esikler': th, 'hiz_ms': round(test.speed['inference'], 1)}
    rows.append({'model': name, **{k: results[name][k] for k in ('mAP50', 'mAP50-95', 'precision', 'recall')},
                 **{f'AP50 {c}': v for c, v in per_class.items()}, 'çıkarım ms': results[name]['hiz_ms']})
    # Uygulamanın okuyacağı biçimde eşik dosyası
    tag = name.split()[0]
    json.dump({'esikler': {c: v['esik'] for c, v in th.items()},
               'kaynak': f'RDD2022 doğrulama seti, sınıf başına en yüksek F1 ({name})'},
              open(f'{OUT}/esikler_{tag}.json', 'w'), indent=2, ensure_ascii=False)

table = pd.DataFrame(rows).set_index('model')
display(table)
print('\nSınıf bazlı eşikler (doğrulama setinde en yüksek F1):')
for name, r in results.items():
    print(' ', name, {c: v['esik'] for c, v in r['esikler'].items()})
json.dump(results, open(f'{OUT}/kiyas_v1_v2.json', 'w'), indent=2, ensure_ascii=False)
print('\nDrive:', OUT, '→ best_v2.pt, esikler_v1.json, esikler_v2.json, kiyas_v1_v2.json')

## 5. Sabah yapılacaklar
Drive'daki `yol-hasar-tespit/v2/` klasöründen şu dosyaları indirip `C:\Projeler\yol-hasar-tespit\models\` içine koyun:
- `best_v2.pt`
- `esikler_v1.json` ve `esikler_v2.json`
- `kiyas_v1_v2.json`

Sonra Claude'a "dosyaları koydum" yazın. v1 ile v2 Bursa test setinde ölçülür, daha iyi olan uygulamaya alınır.